<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Setup_backtest_logging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One-time setup: automatic backtest log

Run this notebook **once** after adding a Colab Secret named `GITHUB_TOKEN` with Contents read/write access to `natdanaiii/Trading`. It patches `Grid_trading.ipynb` so future runs automatically update `logs/latest_backtest_log.json`. The token itself is never written to GitHub.

In [ ]:
import base64, json, textwrap, requests
from google.colab import userdata

TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'natdanaiii/Trading'
BRANCH = 'main'
NOTEBOOK_PATH = 'Grid_trading.ipynb'
API = f'https://api.github.com/repos/{REPO}/contents/{NOTEBOOK_PATH}'
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28',
}

r = requests.get(API, headers=HEADERS, params={'ref': BRANCH}, timeout=30)
r.raise_for_status()
meta = r.json()
nb = json.loads(base64.b64decode(meta['content']).decode('utf-8'))

# Remove an older copy if this setup is rerun.
clean = []
skip_next = False
for cell in nb['cells']:
    if skip_next:
        skip_next = False
        continue
    src = cell.get('source', '')
    src = ''.join(src) if isinstance(src, list) else src
    if cell.get('cell_type') == 'markdown' and src.startswith('## 12. Export Latest Backtest Log to GitHub'):
        skip_next = True
        continue
    clean.append(cell)
nb['cells'] = clean

md = {
    'cell_type': 'markdown',
    'metadata': {},
    'source': '## 12. Export Latest Backtest Log to GitHub\n\nThis cell automatically updates `logs/latest_backtest_log.json` after a successful run. It uses the Colab Secret `GITHUB_TOKEN`; the token is never stored in the notebook or log.',
}

logger = r'''import base64
import hashlib
import json
import marshal
from datetime import datetime, timezone
import requests
from google.colab import userdata

GITHUB_REPO = 'natdanaiii/Trading'
GITHUB_BRANCH = 'main'
GITHUB_LOG_PATH = 'logs/latest_backtest_log.json'

def _json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, pd.Period):
        return str(value)
    if pd.isna(value):
        return None
    return value

def _records(df):
    return [{k: _json_safe(v) for k, v in row.items()} for row in df.to_dict(orient='records')]

engine_hash = hashlib.sha256(
    marshal.dumps(build_excel_grid_table.__code__)
    + marshal.dumps(run_grid_backtest.__code__)
).hexdigest()[:16]

open_grid_df = df_grid_state.loc[df_grid_state['holding']].copy()
log_payload = {
    'run_info': {
        'run_time_utc': datetime.now(timezone.utc).isoformat(),
        'engine_hash': engine_hash,
        'symbol': SYMBOL,
        'start_date': START_DATE,
        'end_date': END_DATE,
        'timeframe': '1m',
        'data_rows': int(len(df_1m)),
        'first_data_time': _json_safe(df_1m['open_time'].iloc[0]),
        'last_data_time': _json_safe(df_1m['open_time'].iloc[-1]),
    },
    'parameters': {
        'capital': float(BACKTEST_CAPITAL),
        'floor': float(BACKTEST_FLOOR),
        'ceiling': float(BACKTEST_CEILING),
        'gap': float(BACKTEST_GAP),
        'price_rounding': float(PRICE_ROUNDING),
        'number_of_grids': int(NUMBER_OF_GRIDS),
        'capital_per_grid': float(CAPITAL_PER_LEVEL),
        'buy_fee': float(BUY_FEE),
        'sell_fee': float(SELL_FEE),
        'historical_low': float(historical_low),
        'historical_high': float(historical_high),
    },
    'verification': {
        'status': 'PASS' if verification_failed.empty else 'FAIL',
        'tests': _records(df_verification_report),
    },
    'system_audit': {
        'status': 'PASS' if failed_system_tests.empty else 'FAIL',
        'tests': _records(df_system_test_log),
    },
    'backtest_summary': {k: _json_safe(v) for k, v in backtest_summary.items()},
    'reconciliation': {
        'net_cash_movement': _json_safe(net_cash_movement),
        'expected_final_cash': _json_safe(BACKTEST_CAPITAL + net_cash_movement),
        'total_grid_cashflow': _json_safe(df_grid_cashflow['grid_cashflow'].sum()),
        'portfolio_gain': _json_safe(portfolio_gain),
    },
    'monthly_grid_cashflow': _records(df_grid_cashflow_monthly),
    'monthly_portfolio_pnl': _records(df_portfolio_pnl_monthly),
    'trading_diagnostics': {
        'buy_count': int(df_trade_log['side'].eq('BUY').sum()) if len(df_trade_log) else 0,
        'sell_count': int(df_trade_log['side'].eq('SELL').sum()) if len(df_trade_log) else 0,
        'completed_cycles': int(backtest_summary['completed_cycles']),
        'open_positions': int(backtest_summary['open_positions']),
        'first_10_events': _records(df_trade_log.head(10)) if len(df_trade_log) else [],
        'last_10_events': _records(df_trade_log.tail(10)) if len(df_trade_log) else [],
        'open_grid_positions': _records(open_grid_df[['level','buy_price','sell_price','capital_per_level','base_amount','buy_time']]) if len(open_grid_df) else [],
    },
}

log_json = json.dumps(log_payload, indent=2, ensure_ascii=False, allow_nan=False)
github_token = userdata.get('GITHUB_TOKEN')
api_url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_LOG_PATH}'
headers = {
    'Authorization': f'Bearer {github_token}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28',
}
existing = requests.get(api_url, headers=headers, params={'ref': GITHUB_BRANCH}, timeout=30)
payload = {
    'message': 'Update latest backtest log',
    'content': base64.b64encode(log_json.encode('utf-8')).decode('ascii'),
    'branch': GITHUB_BRANCH,
}
if existing.status_code == 200:
    payload['sha'] = existing.json()['sha']
elif existing.status_code != 404:
    existing.raise_for_status()
uploaded = requests.put(api_url, headers=headers, json=payload, timeout=30)
uploaded.raise_for_status()
print('===== GITHUB BACKTEST LOG =====')
print('Status      : UPLOADED')
print(f'Path        : {GITHUB_LOG_PATH}')
print(f'Engine hash : {engine_hash}')
print(f'Commit SHA  : {uploaded.json()["commit"]["sha"]}')
'''

nb['cells'].extend([md, {'cell_type':'code','execution_count':None,'metadata':{},'outputs':[],'source':logger}])
new_content = json.dumps(nb, separators=(',', ':'), ensure_ascii=False).encode('utf-8')
payload = {
    'message': 'Add automatic GitHub backtest log export',
    'content': base64.b64encode(new_content).decode('ascii'),
    'sha': meta['sha'],
    'branch': BRANCH,
}
u = requests.put(API, headers=HEADERS, json=payload, timeout=30)
u.raise_for_status()
print('Grid_trading.ipynb patched successfully.')
print('Commit:', u.json()['commit']['sha'])
print('Now open Grid_trading.ipynb and Run all.')